# 教程 5：decode 时在线训练 —— 跨步与跨请求的训练目标

**场景**：一个 LoRA 式的头在 decode 途中逐步训练，它的训练目标不止"当前步"：

1. **当前步**最后一层的 hidden（同层阶梯映射）；
2. **下一步**第 9 层的 hidden（这一步写下自己的 h9，下一步拿真实的 h9 当目标）；
3. **上一个 VIP 请求**（同一路由）最终的第 9 层 hidden（跨请求的目标）。

前两个目标跨**步**，第三个跨**请求**。支撑它们的只有一对概念（教程 4 已见过）：

- **memory 字段**：`RequestState(..., memory={...})` 声明不参与优化、但跟随训练
  发布的普通字段；
- **捕获**：训练输入上的 `pllm.Capture("head", "prev_h9")` 表示发布时把该输入
  最后一行写进 memory；
- **输入版本语义**：memory 读到的是**上一步发布**的值 —— 这就是"跨步"的全部含义；
  而 lineage 跨请求存活（route 键）就是"跨请求"的全部含义。

## Workload：三个目标写在同一个 loss 里

逐行看 `learn` 的输入：

- `h9` / `h9_anchor`：同一个第 9 层 hidden 读两次，分别捕获到 `prev_h9`（每步覆盖）
  和 `anchor`（每步覆盖，请求结束后留存为终态）；
- `prev_h9` / `anchor`（`Parameter` 读）：memory 字段的**输入版本** —— 上一步发布的值；
- 三个 loss 项分别对应三个目标。

`predict` 把 `anchor` 也加进引导：VIP 的终态直接参与后续请求的采样。

In [1]:
from typing import Annotated
import torch
import pllm
from vllm import SamplingParams
from pllm.config import VllmDType, ModelConfig


class DecodeTrainer(pllm.Workload):
    """decode 途中训练映射头 w，三类目标同时上线。"""

    head = pllm.RequestState(
        {"w": lambda m: (m.hidden_size, m.hidden_size)},
        dtype=torch.float32,
        optimizer=pllm.AdamW(lr=1e-3),
        memory={"prev_h9": lambda m: (m.hidden_size,),
                "anchor": lambda m: (m.hidden_size,)},
        keys=(pllm.RequestKeyField.ROUTE,))

    def ready(self, versions: Annotated[torch.Tensor, pllm.StateVersion("head")]) -> torch.Tensor:
        return versions > 0

    @pllm.on(pllm.Phase.BEFORE_SAMPLE, rows="ready")
    @pllm.guide_sampling(beta=0.5)
    def predict(self, h9: Annotated[torch.Tensor, pllm.Hidden(9)],
                w: Annotated[torch.Tensor, pllm.Parameter("head", "w")],
                anchor: Annotated[torch.Tensor, pllm.Parameter("head", "anchor")],
                versions: Annotated[torch.Tensor, pllm.StateVersion("head")]) -> torch.Tensor:
        return torch.bmm(h9.float().unsqueeze(1), w).squeeze(1) + 0.5 * anchor

    @pllm.on(pllm.Phase.AFTER_SAMPLE)
    @pllm.train(state="head", reduction="state_mean")
    def learn(self,
              h9: Annotated[torch.Tensor, pllm.Hidden(9), pllm.Capture("head", "prev_h9")],
              h9_anchor: Annotated[torch.Tensor, pllm.Hidden(9), pllm.Capture("head", "anchor")],
              h_last: Annotated[torch.Tensor, pllm.Hidden(-1)],
              w: Annotated[torch.Tensor, pllm.Parameter("head", "w")],
              prev_h9: Annotated[torch.Tensor, pllm.Parameter("head", "prev_h9")],
              anchor: Annotated[torch.Tensor, pllm.Parameter("head", "anchor")],
              groups: Annotated[torch.Tensor, pllm.StateIndex("head")]) -> torch.Tensor:
        prediction = torch.bmm(h9.float().unsqueeze(1), w).squeeze(1)
        loss_now = (prediction - h_last.float()).square().mean(1)  # 目标 1：当前步最后一层
        loss_next = (torch.bmm(prev_h9.unsqueeze(1), w).squeeze(1)
                     - h9.float()).square().mean(1)                # 目标 2：上一步预测 → 这一步的真值
        loss_vip = (prediction - anchor).square().mean(1)          # 目标 3：向上一请求的终态对齐
        losses = loss_now + loss_next + loss_vip
        counts = torch.bincount(groups).to(losses)
        return losses.new_zeros(counts.shape).index_add(0, groups, losses) / counts

/workspace/pllm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 怎么看见这三条通道

训练任务在 trace 选择器开启时记录逐步 `loss`。设计三次 generate：

1. **VIP**（route="vip"，新 lineage）：首步的 `prev_h9`/`anchor` 都是零页，
   后续步读到自己上一步的捕获 —— 逐步 loss 应当呈现明显的轨迹变化；
2. **续接**（route="vip"）：加入 VIP 的 lineage，首步就读到 VIP 写下的两个
   memory 字段和训练过的 `w` —— 首步 loss 必须与全新 route 不同；
3. **两个全新 route**（"fresh" / "fresh2"）：零状态确定性 —— 同一 prompt 的
   首步 loss 必须**完全相等**。

In [2]:
from pllm.execution.trace import TraceSelection

llm = pllm.make_llm(workload=DecodeTrainer(), config=ModelConfig(
    "Qwen/Qwen2.5-0.5B-Instruct", VllmDType.BFLOAT16, 96, 4, 96, 0.2, seed=7))
plan = llm.attachment.inputs.plan
learn_index = next(i for i, task in enumerate(plan.semantic_graph.tasks)
                   if str(task.symbol) == "learn")
learn_id = next(task.task_id for task in plan.schedule_physical.tasks
                if task.task_id == learn_index)


def run(prompt, route):
    """一次 generate 的逐步 loss 与文本（trace 游标切片）。"""
    before = llm.trace_snapshot()
    cursor = before.dropped + len(before.events)
    llm.observe(TraceSelection(tasks=(learn_id,), tensors=True, max_tensor_bytes=32768))
    try:
        outputs = llm.generate([prompt], SamplingParams(max_tokens=6,
            temperature=0.0, ignore_eos=True), use_tqdm=False, routes=(route,))
        snapshot = llm.trace_snapshot()
        fresh = snapshot.events[max(0, cursor - snapshot.dropped):]
        losses = [(e.step_id, e.tensor().item()) for e in fresh if e.tensor_name == "loss"]
        return losses, outputs[0].outputs[0].text
    finally:
        llm.observe(TraceSelection())


vip_losses, vip_text = run("Important notice for alice@example.com: the VIP briefing", "vip")
print("VIP 逐步 loss:", [f"{v:.3f}" for _, v in vip_losses])
print("VIP 文本:", vip_text)

follow_losses, follow_text = run("Summarize the briefing", "vip")
fresh_losses, fresh_text = run("Summarize the briefing", "fresh")
fresh2_losses, _ = run("Summarize the briefing", "fresh2")
print("续接（同 route）首步 loss:", follow_losses[0][1], "文本:", follow_text)
print("新 route 首步 loss:", fresh_losses[0][1], "文本:", fresh_text)

# 跨步通道：VIP 首步的 prev_h9 是零，后续步读到自己上一步的捕获。
assert vip_losses[0][1] != vip_losses[-1][1]
# 跨请求通道：续接请求读到 VIP 的 memory 与训练累积，首步 loss 不同。
assert follow_losses[0][1] != fresh_losses[0][1]
# 零状态确定性：两个全新 route 的首步 loss 完全一致。
assert fresh_losses[0][1] == fresh2_losses[0][1]

d = llm.worker_state().diagnostics
print("losses:", d.loss_count, "updates:", d.update_count, "fallback:", d.fallback_count)
llm.close()

2026-09-10 13:15:58,836	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 09-10 13:15:59 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 96, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 96, 'max_num_seqs': 4, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 13:16:00 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 13:16:01 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 13:16:01 [model.py:1883] Using max model len 96


WARNING 09-10 13:16:01 [arg_utils.py:2620] This model does not officially support disabling chunked prefill. Disabling this manually may cause the engine to crash or produce incorrect outputs.


... (27 outputs left out)

INFO 09-10 13:16:13 [hf.py:540] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


WARNING 09-10 13:16:13 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _bmm_outer_product_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


VIP 逐步 loss: ['6.317', '5.714', '4.889', '5.649', '6.318']
VIP 文本:  is now available. The briefing


续接（同 route）首步 loss: 8.644431114196777 文本:  on the impact of climate change
新 route 首步 loss: 8.6561918258667 文本:  on the impact of climate change
losses: 20 updates: 20 fallback: 0


## 要点回顾

- 三类训练目标 = 三类输入来源：当前步 hidden、memory（上一步）、memory（上一请求）。
  框架不需要知道"目标"的概念 —— 全是普通的输入声明。
- `Capture` 写在输入上，发布时落进 memory；读取永远是输入版本（上一步发布的值）。
- 跨请求 = lineage 跨请求存活（route 键），不是任何请求侧的历史对象。
- 验证通道是否工作，最直接的办法是打开 trace 选择器看逐步 loss，
  再配合"全新 route 必须等于全新 route"的零状态确定性检查。

至此五篇场景教程结束。回 [README](../../README.md) 或查
[词汇表](../reference/glossary.md) 巩固术语。